# Модуль 05 · Урок 3. Скользящие, нарастающие и экспоненциальные окна

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Rolling — статистика локального окна
`rolling(window=3)` считает статистику по трём соседним наблюдениям: текущему и двум предыдущим. Обычно первые 2 элемента будут NA, если `min_periods` не изменять. Для плавных графиков используется `rolling(3,min_periods=1)`, когда первые значения считаются по меньшему числу точек. Важная ловушка: окно из 3 строк не тождественно окну из 3 дней, если календарь имеет пропуски.

In [ ]:
import pandas as pd
load = pd.Series([10,15,25,20,5])
print(load.rolling(3,min_periods=1).mean())

## 2. Expanding — окно от начала до текущего значения
`expanding()` на каждом шаге расширяет диапазон с первого наблюдения: накопительное среднее на момент t. Это позволяет рассчитать исторический baseline или онлайн-оценку среднего без использования будущих наблюдений. Сравните с `.expanding().max()` и `.cumsum()` — разные агрегаты на одном типе окна.

In [ ]:
print(load.expanding(min_periods=1).mean())

## 3. EWM — больший вес недавних событий
`ewm(span=3,adjust=False).mean()` строит экспоненциально сглаженный ряд, где старые наблюдения постепенно теряют вес. `span` управляет скоростью затухания; чем он меньше, тем сильнее влияние последней точки. Результат не равен обычной скользящей средней. Для воспроизводимости фиксируйте `span` и `adjust`, иначе сравнение графиков может быть неверным.

In [ ]:
print(load.ewm(span=3,adjust=False).mean())

## 4. Окна внутри групп и утечка будущего
Когда в данных несколько устройств или магазинов, окна должны считаться **внутри каждой группы** и после правильной сортировки по времени. В задачах прогнозирования особенно опасно использовать статистику, куда попали будущие наблюдения: это называется data leakage. Для признака «среднее до текущего события» требуется сначала `shift(1)`, затем `rolling`.

**Проверьте себя:** чем rolling(3) отличается от rolling('3D')? Что такое expanding? Почему важно не включать будущие события в исторический признак?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m5_11 · rolling mean

Получите скользящее среднее Series по `window` последним значениям с `min_periods=1`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `values` — `pd.Series` (индекс и значения определены в ячейке примера).
- `window` — входной параметр (значение видно в примере); в этом примере: `3`.


**Выполнение:** используйте готовые переменные `values`, `window`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_11: используются переменные values, window

values = pd.Series([2, 8, 4, 10, 12], index=[0, 1, 2, 3, 4])

window = 3

print("values — демонстрационные данные:")

display(values)


# Снимок правильного результата для автопроверки.
_expected_m5_11 = expected_for("m5_11", values=values, window=window)


In [ ]:
# m5_11: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: values, window
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 5 значений; имя серии: `None`.

```text
0    2.000000
1    5.000000
2    4.666667
3    7.333333
4    8.666667
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_11", result, _expected_m5_11)

### m5_12 · past only mean

Постройте признак: среднее из `window` ПРЕДЫДУЩИХ наблюдений, не включая текущее. Для первой строки будет NA.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `values` — `pd.Series` (индекс и значения определены в ячейке примера).
- `window` — входной параметр (значение видно в примере); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `values`, `window`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_12: используются переменные values, window

values = pd.Series([4.0, 8.0, 2.0, 6.0], index=[0, 1, 2, 3])

window = 2

print("values — демонстрационные данные:")

display(values)


# Снимок правильного результата для автопроверки.
_expected_m5_12 = expected_for("m5_12", values=values, window=window)


In [ ]:
# m5_12: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: values, window
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 4 значений; имя серии: `None`.

```text
0    NaN
1    4.0
2    6.0
3    5.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_12", result, _expected_m5_12)

### m5_13 · expanding max

Получите нарастающий максимум Series при расширяющемся окне с min_periods=1.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `values` — `pd.Series` (индекс и значения определены в ячейке примера).


**Выполнение:** используйте готовые переменные `values`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_13: используются переменные values

values = pd.Series([3, 1, 7, 2], index=[0, 1, 2, 3])

print("values — демонстрационные данные:")

display(values)


# Снимок правильного результата для автопроверки.
_expected_m5_13 = expected_for("m5_13", values=values)


In [ ]:
# m5_13: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: values
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 4 значений; имя серии: `None`.

```text
0    3.0
1    3.0
2    7.0
3    7.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_13", result, _expected_m5_13)

### m5_14 · ewm score

Выполните экспоненциальное сглаживание Series с `span` и `adjust=False`, получите Series.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `values` — `pd.Series` (индекс и значения определены в ячейке примера).
- `span` — входной параметр (значение видно в примере); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `values`, `span`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_14: используются переменные values, span

values = pd.Series([9.0, 7.0, 3.0, 11.0], index=[0, 1, 2, 3])

span = 2

print("values — демонстрационные данные:")

display(values)


# Снимок правильного результата для автопроверки.
_expected_m5_14 = expected_for("m5_14", values=values, span=span)


In [ ]:
# m5_14: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: values, span
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 4 значений; имя серии: `None`.

```text
0    9.000000
1    7.666667
2    4.555556
3    8.851852
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_14", result, _expected_m5_14)

### m5_15 · group rolling

Сортируйте ledger по shop/date и добавьте `moving_avg` — скользящее среднее amount по 2 последним записям **каждого магазина** с min_periods=1.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `date`, `shop`, `amount`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_15: используются переменные df

df = pd.DataFrame({
    'date': ['2026-01-01', '2026-01-02', '2026-01-04', '2026-01-07', '2026-01-08'],
    'shop': ['A', 'A', 'A', 'B', 'B'],
    'amount': [10.0, 25.0, 30.0, 40.0, 15.0],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_15 = expected_for("m5_15", df=df)


In [ ]:
# m5_15: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 4 столбца; колонки: `date`, `shop`, `amount`, `moving_avg`.

```text
         date shop  amount  moving_avg
0  2026-01-01    A    10.0        10.0
1  2026-01-02    A    25.0        17.5
2  2026-01-04    A    30.0        27.5
3  2026-01-07    B    40.0        40.0
4  2026-01-08    B    15.0        27.5
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_15", result, _expected_m5_15)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()